In [1]:
import pandas as pd

import lsdb


In [2]:
cat_a = lsdb.open_catalog('tests/data/small_sky_order1_collection')

In [3]:
cat_a

,id,ra,dec,ra_error,dec_error
npartitions=4,,,,,
"Order: 1, Pixel: 44",int64[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],int64[pyarrow]
"Order: 1, Pixel: 45",...,...,...,...,...
"Order: 1, Pixel: 46",...,...,...,...,...
"Order: 1, Pixel: 47",...,...,...,...,...


In [4]:
def rename_cols(df, names_in, names_out):
    """df = rename_cols(df, ['ra', 'dec'], ['my_ra', 'my_dec'])"""
    for name_in, name_out in zip(names_in, names_out):
        df[name_out] = df[name_in]
    col_names = [col for col in df.columns if col not in names_in]
    return df[col_names]

In [5]:
# Scenario A: Creating an invalid catalog via from_dataframe() / DataFrameCatalogLoader

# Succeeds
cat_b = lsdb.from_dataframe(rename_cols(cat_a.compute(), ['ra', 'dec'], ['my_ra', 'my_dec']), 
    ra_column='my_ra', dec_column='my_dec')

# Fails
# Error happens during catalog creation
try:
    cat_b = lsdb.from_dataframe(rename_cols(cat_a.compute(), ['ra', 'dec'], ['my_ra', 'my_dec']))
except Exception as e:
    print(repr(e))

Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError("No column found for 'ra' (required). You can supply ra/dec column names using the arguments `ra_column`, `dec_column`.")


In [6]:
# Scenario A2: ambiguous column matches
# TODO

In [8]:
# Scenario B1: Invalid catalog via map_partitions()

# This creates an invalid catalog: the hc_structure contains the old ra and dec column names,
# but now there are no columns with those names.
cat_b = cat_a.map_partitions(rename_cols, ['ra', 'dec'], ['my_ra', 'my_dec'])

# But we don't get an error until we try to use the catalog:
try:
    cat_a.crossmatch(cat_b)
except Exception as e:
    print(repr(e))

# Possible fix: make map_partitions() warn when it modifies ra or dec columns

ValueError('right table must have column ra')


/Users/heather/repos/lsdb/src/lsdb/catalog/catalog.py:410: FutureWarning: The default suffix behavior will change from applying suffixes to all columns to only applying suffixes to overlapping columns in a future release.To maintain the current behavior, explicitly set `suffix_method='all_columns'`. To change to the new behavior, set `suffix_method='overlapping_columns'`.
  warnings.warn(


In [9]:
# Scenario B2: Invalid catalog via map_partitions()

# map_partitions(..., compute_single_partition=True) now errors when ra/dec columns change
cat_b = cat_a.map_partitions(rename_cols, ['ra', 'dec'], ['my_ra', 'my_dec'], compute_single_partition=True)

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: 'ra' not found in result. map_partitions() must not change names of ra or dec columns 'ra', 'dec'.

In [11]:
# Scenario B3: Invalid catalog via map_partitions()

def my_bad_function(df, col_name):
    df[col_name] = df[col_name] + 1
    return df

# map_partitions(..., compute_single_partition=True) errors when ra/dec values change
for col_name in ['ra', 'dec']:
    try:
        cat_b = cat_a.map_partitions(my_bad_function, col_name, compute_single_partition=True)
    except Exception as e:
        print(repr(e))



Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError("ra/dec values have changed. map_partitions() must not change values of ra or dec columns 'ra', 'dec'.")


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError("ra/dec values have changed. map_partitions() must not change values of ra or dec columns 'ra', 'dec'.")
